# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [26]:
import warnings
warnings.filterwarnings("ignore")
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [27]:
from dotenv import load_dotenv
import tiktoken

# Load environment variables
load_dotenv(override=True)

True

In [28]:
# Search tool
tavily_search = TavilySearchResults()

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Tokenizer
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

In [29]:
def safe_search(query: str) -> str:
    """
    Perform a Tavily search and trim the result
    to avoid exceeding the model context window.
    """
    result = tavily_search.invoke({"query": query})

    if isinstance(result, list):
        result_text = " ".join(str(item) for item in result)
    elif isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)

    return encoding.decode(tokens[:3500])


tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Search the web for up-to-date information."
    )
]

In [44]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)

response = agent.invoke(
    {
        "input": (
            "What are the latest AI research developments "
            "in retrieval-augmented generation?"
        )
    }
)

print(response["output"])



> Entering new AgentExecutor chain...
I should search for the latest information on AI research developments specifically related to retrieval-augmented generation, as this is a rapidly evolving field. 

Action: TavilySafeSearch  
Action Input: "latest AI research developments in retrieval-augmented generation 2023"  
Observation: {'title': 'awesome-generative-ai-guide/research_updates/rag_research_table.md at main · aishwaryanr/awesome-generative-ai-guide · GitHub', 'url': 'https://github.com/aishwaryanr/awesome-generative-ai-guide/blob/main/research_updates/rag_research_table.md', 'content': "| Learning to Filter Context for Retrieval-Augmented Generation | The FILCO method is introduced to enhance the quality of context provided to generation models in retrieval-augmented systems. By identifying useful context and training context filtering models, FILCO aims to mitigate issues arising from irrelevant passages during generation. Experimental results across various knowledge-intens

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# Your prompt here
prompt_1 = "You are operating in the current context of Generative AI (GenAI) and healthcare. Retrieve a comprehensive and thorough overview over new GenAI applications in healthcare as of the beginning of 2026. List all applications separately and give brief descriptions of involved models and methods, as well as the achieved benefits where available. Also highlight experience reports by healthcare professionals involved in introducing, developing or testing the applications, as well as patient feedback, benefits and further research needs. Limit yourself to a maximum of 5 sentences for each application and choose a maximum of the top ten applications. In case there seems to be significantly more or significantly less information available, report this accordingly. Provide a structured response in Markdown format."

# Run it
response_1 = agent.run(prompt_1)




> Entering new AgentExecutor chain...
I need to gather up-to-date information on new Generative AI applications in healthcare as of the beginning of 2026. This will involve searching for recent developments, applications, models, methods, and feedback from healthcare professionals and patients. 

Action: TavilySafeSearch  
Action Input: "Generative AI applications in healthcare 2026 overview"  
Observation: {'title': '2026 healthcare AI trends: Insights from experts | Wolters Kluwer', 'url': 'https://www.wolterskluwer.com/en/expert-insights/2026-healthcare-ai-trends-insights-from-experts', 'content': '“Generative AI is revolutionizing healthcare, offering transformative capabilities in clinical documentation, decision support, patient communication, and medical education. These tools promise to reduce administrative burden, enhance diagnostic precision, and improve patient engagement—benefits that are already being realized by early adopters across health systems.\n\nBut alongside th

In [46]:
display(Markdown(response_1))

As of 2026, the top Generative AI applications in healthcare include Clinical Documentation Automation, AI-Powered Diagnostics, Personalized Treatment Plans, Patient Communication Enhancement, Operational Efficiency, AI in Medical Education, Predictive Analytics for Preventive Care, Integration of Holistic Approaches, AI-Enhanced Research and Development, and Governance and Ethical Considerations. Each application leverages advanced AI models and methods to improve patient care, operational efficiency, and educational outcomes, with positive feedback from both healthcare professionals and patients.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [45]:
# Your prompt here
prompt_2 = "You are operating in the current context of emerging startups whose business model or core functionality is based on Artificial Intelligence (AI). Retrieve a comprehensive and thorough overview over emerging AI startups and their innovations as of spring 2026. Provide a response in Markdown format, listing a maximum of 10 company names, each followed by a short explanation of their AI-related innovation, product highlights and / or unique business models. Clearly display sources for each listed company. Do not invent any information. Indicate whether and where there is a lack of available information."

# Run it
response_2 = agent.invoke(
    {"input": prompt_2}
)

display(Markdown(response_2["output"]))



> Entering new AgentExecutor chain...
I need to gather up-to-date information on emerging AI startups and their innovations as of spring 2026. This requires a web search to find relevant and current data.

Action: TavilySafeSearch  
Action Input: "emerging AI startups innovations spring 2026"  
Observation: {'title': 'The most innovative artificial intelligence companies of 2026', 'url': 'https://www.fastcompany.com/91495412/artificial-intelligence-most-innovative-companies-2026', 'content': 'Many of the strongest startups are started by people who had a personal need for the company’s product. That’s the case with GC AI, whose name refers to AI for general counsels, the corporate world’s top in-house lawyers. GC AI was cofounded by Cecilia Ziniti, who was general counsel in Amazon’s Alexa division, at the coding assistant company Replit, and at the autonomous driving company Cruise. [...] Li started World Labs with well-known AI researchers Justin Johnson, Christoph Lassner, and Ben

The above list provides a comprehensive overview of emerging AI startups and their innovations as of spring 2026, along with sources for further reading.

In [47]:
display(Markdown(response_2["output"]))


The above list provides a comprehensive overview of emerging AI startups and their innovations as of spring 2026, along with sources for further reading.

### Exercise 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [48]:
# Your prompt here
prompt_3 = "You are operating in the current context of Generative Artificial Intelligence (GenAI), the underlying Retrieval-Augmented Generation (RAG) technology and the commercial context of these products. Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro. Retrieve a comprehensive and thorough overview over each aspect for both tech products and explain each aspect in 1-2 sentences. Display a clear juxtaposition of the two and give an indication of the strengths and weaknesses of each product regarding each category. Provide a response in Markdown format. Clearly display sources for each listed company. Do not invent any information. Indicate whether and where there is a lack of available information."

# Run it
response_3 = agent.run(prompt_3)




> Entering new AgentExecutor chain...
I need to gather up-to-date information on OpenAI's ChatGPT Team and Anthropic's Claude Pro, focusing on their key features, pricing, and reviews. This will help me create a comprehensive comparison of the two products. 

Action: TavilySafeSearch  
Action Input: "OpenAI ChatGPT Team features pricing reviews 2023"  
Observation: {'title': "ChatGPT for teams is here. Here's my review Last week, OpenAI ...", 'url': 'https://www.linkedin.com/posts/accessible-ai_chatgpt-for-teams-is-here-heres-my-review-activity-7152271601139425280-cEU2', 'content': "ChatGPT for teams is here. Here's my review 👇 Last week, OpenAI launched a new plan: ChatGPT Team, aimed at businesses with 2-149 people. It promises: --> Collaborative workspaces: shared chats, private GPTs --> Data privacy: data won't be used for training --> Usage controls: The ability to control team activity --> Higher usage: Increased to 100 messages/3 hours (currently) --> Early access to new featu

In [ ]:
display(Markdown(response_3))


## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
